# Nonlinear pendulum

$$H(\theta, p) = \frac{p^2}{2 m L^2} - m g L \cos\theta$$

Released from rest at amplitude $\theta_0$, the exact period is $T = 4\sqrt{L/g}\,K(\sin^2(\theta_0/2))$, with $K$ the complete elliptic integral of the first kind. This notebook checks that formula against numerical integration.

**Evidence:** the equations of motion are *computed*; the period formula is *checked numerically*; energy conservation is *stated but not yet proved* in `lean/Hamiltonian/Pendulum.lean`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

# The project root is the nearest folder containing manuscript/.
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "manuscript").is_dir())
GENERATED = ROOT / "manuscript" / "generated"
FIGURES = ROOT / "manuscript" / "figures"
GENERATED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)


def latex_sci(x):
    """Format a number for LaTeX, e.g. 3.2e-11 -> $3.2 \\times 10^{-11}$."""
    mantissa, exponent = f"{x:.1e}".split("e")
    return f"${mantissa} \\times 10^{{{int(exponent)}}}$"

## Hamilton's equations

In [ ]:
theta, p = sp.symbols("theta p", real=True)
m, g, L = sp.symbols("m g L", positive=True)

H = p**2 / (2 * m * L**2) - m * g * L * sp.cos(theta)
theta_dot = sp.diff(H, p)
p_dot = -sp.diff(H, theta)

assert sp.simplify(sp.diff(H, theta) * theta_dot + sp.diff(H, p) * p_dot) == 0

sp.Eq(sp.Symbol(r"\dot\theta"), theta_dot), sp.Eq(sp.Symbol(r"\dot p"), p_dot)

In [ ]:
eom_tex = (
    "\\begin{align}\n"
    f"  \\dot\\theta &= {sp.latex(theta_dot)}, \\\\\n"
    f"  \\dot p &= {sp.latex(p_dot)}.\n"
    "\\end{align}\n"
)
(GENERATED / "pendulum_eom.tex").write_text(eom_tex, encoding="utf-8")
print(eom_tex)

## Period against amplitude

In [ ]:
from scipy.special import ellipk

values = {m: 1.0, g: 9.81, L: 1.0}
rhs = sp.lambdify((theta, p), [theta_dot.subs(values), p_dot.subs(values)], "numpy")
g_val, L_val = values[g], values[L]
T_small = 2 * np.pi * np.sqrt(L_val / g_val)


def upward_zero_crossing(t, y):
    return y[0]


upward_zero_crossing.direction = 1


def period(theta0):
    """Time between successive upward zero crossings, released from rest at theta0."""
    sol = solve_ivp(lambda t, y: rhs(*y), (0, 10 * T_small), [theta0, 0.0],
                    events=upward_zero_crossing, rtol=1e-10, atol=1e-12)
    first, second = sol.t_events[0][:2]
    return second - first


amplitudes = np.linspace(0.05, 0.95 * np.pi, 40)
numerical = np.array([period(a) for a in amplitudes])
exact = 4 * np.sqrt(L_val / g_val) * ellipk(np.sin(amplitudes / 2) ** 2)

worst = np.max(np.abs(numerical / exact - 1))
(GENERATED / "pendulum_period_check.tex").write_text(latex_sci(worst), encoding="utf-8")
print(f"Largest relative difference from the elliptic-integral formula: {worst:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(amplitudes, exact / T_small, label="exact (elliptic integral)")
ax.plot(amplitudes, numerical / T_small, "o", ms=3, label="numerical")
ax.axhline(1, color="grey", lw=0.8, ls="--", label="small-angle")
ax.set_xlabel(r"amplitude $\theta_0$")
ax.set_ylabel(r"$T / 2\pi\sqrt{L/g}$")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "pendulum_period.pdf")
plt.show()